# Filter initial dataset for plots


The `LeadVariantEffect` dataset has to be filtered for downstream analysis.

For the downstream analysis of MAF, rescaled effect size and variant effects we need to filter the `LeadVariantEffect` dataset to:

1. Limit the dataset to `gwas, cis-pqtl and eqlt` datasets.
2. Split the GWAS studies into two types:
   - measurements (continuous traits)
   - diseases (binary traits bound to therapeutic areas)
3. Apply the `replicated` mask to the GWAS-measurements, GWAS-diseases and molecular QTL datasets.
4. Apply the `qualified` mask to the GWAS-measurements, GWAS-diseases.
5. Apply Posterior Inclusion Probability (PIP) filter to all lead variants, keeping only those with PIP >= 0.1.

After the filtering applied check how many variants are left in the dataset and compute statistics on rescaled effect size & MAF.

5. Ensure there is no lead variants with MAF >= 0.01 and is not empty
6. Ensure there is no lead variants with absolute value of rescaled effect size <= 3 and is not empty

The resulting dataset should be split into two datasets:

- lead variant effects with MAF >= 0.01 (qualified_lead_variant_effect_maf_filtered)
- lead variant effects with all variants (qualified_lead_variant_effect)


## Data Loading

The data required for the analysis is loaded from the

- `lead variant effect` dataset
- `qualified gwas measurements` dataset
- `qualified gwas diseases` dataset
- `replicated molecular qtls` dataset
- `replicated gwas` dataset


### Data downloading


In [ ]:
gwas_therapeutic_areas_path = "../../data/gwas_therapeutic_areas"
qualifying_gwas_disease_studies_path = "../../data/qualifying_disease_studies"
qualifying_gwas_measurements_studies_path = "../../data/qualifying_measurements_studies"
qualifying_gwas_disease_credible_set_path = "../../data/qualifying_disease_credible_sets"
qualifying_gwas_measurements_credible_set_path = "../../data/qualifying_measurement_credible_sets"
lead_variant_effect_dataset_path = "../../data/lead_variant_effect"

qualified_lead_variant_effect_path = "../../data/qualified_lead_variant_effect"
qualified_lead_variant_effect_maf_filtered_path = "../../data/qualified_lead_variant_effect_maf_filtered"
replicated_molqtls_path = "../../data/replicated_molqtl_credible_sets"
replicated_gwas_path = "../../data/replicated_gwas_credible_sets"
replicated_credible_sets_path = "../../data/replicated_credible_sets"


In [ ]:
!gcloud storage rsync -r --delete-unmatched-destination-objects gs://open-targets-data-releases/25.06/output/study ../../data/study


zsh:1: command not found: gcloud


In [35]:
!gcloud storage rsync -r --delete-unmatched-destination-objects  gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/gwas_therapeutic_areas $gwas_therapeutic_areas_path
!gcloud storage rsync -r --delete-unmatched-destination-objects gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/qualifying_studies $qualifying_gwas_disease_studies_path
!gcloud storage rsync -r --delete-unmatched-destination-objects  gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/qualifying_measurements $qualifying_gwas_measurements_studies_path
!gcloud storage rsync -r --delete-unmatched-destination-objects  gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/qualifying_credible_sets $qualifying_gwas_disease_credible_set_path
!gcloud storage rsync -r --delete-unmatched-destination-objects  gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/qualifying_measurement_credible_sets $qualifying_gwas_measurements_credible_set_path
!gcloud storage rsync -r --delete-unmatched-destination-objects gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/list_of_molqtls_replicated_CSs.parquet $replicated_molqtls_path
!gcloud storage rsync -r --delete-unmatched-destination-objects gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/list_of_gwas_replicated_CSs.parquet $replicated_gwas_path


At file://../../data/gwas_therapeutic_areas/**, worker process 3482 thread 8797626112 listed 3...
At gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/gwas_therapeutic_areas/**, worker process 3482 thread 8797626112 listed 3...
  Completed files 0 | 0B                                                       


Updates are available for some Google Cloud CLI components.  To install them,
please run:
  $ gcloud components update

At file://../../data/qualifying_disease_studies/**, worker process 3589 thread 8797626112 listed 3...
At gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/qualifying_studies/**, worker process 3589 thread 8797626112 listed 3...
  Completed files 0 | 0B                                                       
At file://../../data/qualifying_measurements_studies/**, worker process 3684 thread 8797626112 listed 3...
At gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/qualifying_measurements/**, worker process 3684 thread 8797626112 lis

### Data reading


In [ ]:
from gentropy.common.session import Session
from gentropy.dataset.study_locus import StudyLocus
from pyspark.sql import functions as f

from manuscript_methods.datasets import LeadVariantEffect
from manuscript_methods.locus_statistics import LocusStatistics
from manuscript_methods.rescaled_beta import RescaledStatistics
from manuscript_methods.study_statistics import StudyStatistics, StudyType


Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [ ]:
session = Session(extended_spark_conf={"spark.driver.memory": "40G"})
full_lve = LeadVariantEffect.from_parquet(session=session, path=lead_variant_effect_dataset_path)
qualified_gwas_measurements_lve = LeadVariantEffect.from_parquet(
    session=session, path=qualifying_gwas_measurements_credible_set_path
)
qualified_gwas_disease_lve = LeadVariantEffect.from_parquet(
    session=session, path=qualifying_gwas_disease_credible_set_path
)
replicated_molqtls = session.spark.read.parquet(replicated_molqtls_path)
replicated_gwas = session.spark.read.parquet(replicated_gwas_path)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/02 22:31:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
full_lve.df.show(1)


+--------------+--------------------+--------------------+----------+------+------------+---------------------+--------------------+-----------------+----------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------------------+------------------------+
|     variantId|             variant|        studyLocusId|   studyId|geneId|originalBeta|originalStandardError|     locusStatistics|finemappingMethod|isTransQtl|       variantEffect|majorLdPopulation|majorLdPopulationMaf| majorLdPopulationAf|   variantStatistics|     studyStatistics|  rescaledStatistics|leadVariantConsequence|traitFromSourceMappedIds|
+--------------+--------------------+--------------------+----------+------+------------+---------------------+--------------------+-----------------+----------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------

## Analysis steps

1. Filter full lve dataset to keep only cis-pqtl and eqtl datasets.
2. Union with the qualified gwas measurements and diseases datasets lve datasets
3. Apply filtering based on the replicated gwas and molqtls datasets
4. Remove lead variants with MAF == 0.0 and empty MAF and absolute value of rescaled effect size <= 3
5. Filter by PIP >= 0.9
6. Save the filtered dataset to parquet
7. MAF filter >= 0.01
8. Save the filtered datasets


In [ ]:
print(f"Before filtering qtls: {full_lve.df.count():,}")
molqtl_lve = LeadVariantEffect(
    full_lve.df.filter(StudyStatistics().study_type.isin(StudyType.CIS_PQTL, StudyType.EQTL))
)
print(f"After filtering qtls: {molqtl_lve.df.count():,}")


Before filtering qtls: 2,833,758
After filtering qtls: 1,365,531


In [ ]:
print(f"Before union - measurements: {qualified_gwas_measurements_lve.df.count():,}")
print(f"Before union - disease: {qualified_gwas_disease_lve.df.count():,}")
print(f"Before union - molqtl: {molqtl_lve.df.count():,}")

study_stats = StudyStatistics()

# Adjust the study types
qualified_gwas_measurements_lve = LeadVariantEffect(
    qualified_gwas_measurements_lve.df.withColumn(
        study_stats.name, study_stats.transform_study_type(StudyType.GWAS_MEASUREMENT).col
    )
)
qualified_gwas_disease_lve = LeadVariantEffect(
    qualified_gwas_disease_lve.df.withColumn(
        study_stats.name, study_stats.transform_study_type(StudyType.GWAS_DISEASE).col
    )
)


qualified_lve = LeadVariantEffect(
    molqtl_lve.df.unionByName(qualified_gwas_measurements_lve.df).unionByName(qualified_gwas_disease_lve.df)
)
print(f"After union: {qualified_lve.df.count():,}")


Before union - measurements: 450,357
Before union - disease: 70,618
Before union - molqtl: 1,365,531
After union: 1,886,506


In [ ]:
qualified_lve.df.show(1)


+-------------+--------------------+--------------------+--------------------+---------------+------------+---------------------+--------------------+-----------------+----------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------------------+------------------------+
|    variantId|             variant|        studyLocusId|             studyId|         geneId|originalBeta|originalStandardError|     locusStatistics|finemappingMethod|isTransQtl|       variantEffect|majorLdPopulation|majorLdPopulationMaf| majorLdPopulationAf|   variantStatistics|     studyStatistics|  rescaledStatistics|leadVariantConsequence|traitFromSourceMappedIds|
+-------------+--------------------+--------------------+--------------------+---------------+------------+---------------------+--------------------+-----------------+----------+--------------------+-----------------+--------------------+-----------------

In [ ]:
# Filter based on replicated credible sets
print(f"Before filtering replicated credible sets: {qualified_lve.df.count():,}")
print(f"Before filtering replicated GWAS: {replicated_gwas.count():,}")
print(f"Before filtering replicated molQTLs: {replicated_molqtls.count():,}")
replicated_cs_df = (
    replicated_gwas.unionByName(replicated_molqtls).write.mode("overwrite").parquet(replicated_credible_sets_path)
)
replicated_cs = StudyLocus.from_parquet(session=session, path=replicated_credible_sets_path)
cs_count = replicated_cs.df.count()
print(f"Replicated credible sets count: {cs_count:,}")
replicated_qualified_lve = qualified_lve.filter_by_study_locus_id(replicated_cs)
print(f"After filtering replicated credible sets: {replicated_qualified_lve.df.count():,}")


Before filtering replicated credible sets: 1,886,506
Before filtering replicated GWAS: 263,705
Before filtering replicated molQTLs: 1,461,445


Replicated credible sets count: 1,725,150


StudyLocus dimension: 1725150, unique studyLocusId: 1725150


Initial rows: 1886506
Filtered 749416 rows based on the StudyLocus.
Remaining rows: 1137090
After filtering replicated credible sets: 1,137,090


In [ ]:
# Removal of beta outliers and MAF == 0, null
print(f"Before filtering maf and beta outliers: {qualified_gwas_measurements_lve.df.count():,}")
lve_maf_tmp = qualified_gwas_measurements_lve.maf_filter(threshold=None).effect_size_filter()
print(f"After filtering maf and beta outliers: {lve_maf_tmp.df.count():,}")


Before filtering maf and beta outliers: 450,357
After filtering maf and beta outliers: 450,357


In [ ]:
# Removal of beta outliers and MAF == 0, null
print(f"Before filtering maf and beta outliers: {replicated_qualified_lve.df.count():,}")
lve_maf = replicated_qualified_lve.maf_filter(threshold=None).effect_size_filter()
print(f"After filtering maf and beta outliers: {lve_maf.df.count():,}")


Before filtering maf and beta outliers: 1,137,090
After filtering maf and beta outliers: 1,117,120


In [ ]:
# Remove variants with low PIP
pip_threshold = 0.5
locus_stats = LocusStatistics()
print(f"Before filtering PIP outliers: {lve_maf.df.count():,}")
lve = lve_maf.filter(locus_stats.col.getField("leadVariantPIP") >= pip_threshold)
print(f"After filtering PIP outliers: {lve.df.count():,}")


Before filtering PIP outliers: 1,117,120
After filtering PIP outliers: 459,516


In [ ]:
lve.df.write.mode("overwrite").parquet(qualified_lead_variant_effect_path)


In [ ]:
# Apply MAF filter to 0.01
print(f"Before MAF filtering: {lve.df.count():,}")
lve_maf_filtered = lve.maf_filter()
print(f"After MAF filtering: {lve_maf_filtered.df.count():,}")


Before MAF filtering: 459,516


After MAF filtering: 448,966


In [ ]:
lve_maf_filtered.df.write.mode("overwrite").parquet(qualified_lead_variant_effect_maf_filtered_path)


In [ ]:
lve.df.groupBy("studyStatistics.studyType").count().show()


+----------------+------+
|       studyType| count|
+----------------+------+
|        cis-pqtl|  1492|
|            eqtl|354183|
|gwas-measurement| 91147|
|    gwas-disease| 12694|
+----------------+------+



In [ ]:
lve_maf_filtered.df.groupBy("studyStatistics.studyType").count().show()


+----------------+------+
|       studyType| count|
+----------------+------+
|        cis-pqtl|  1470|
|            eqtl|352737|
|gwas-measurement| 82868|
|    gwas-disease| 11891|
+----------------+------+



# Unique number of variants and deduplication


In [ ]:
num_variants = lve.df.filter(f.col("studyStatistics.studyType") == "cis-pqtl").select("variantId").distinct().count()
print(f"Number of unique variants in cis-pqtl: {num_variants}")
num_variants = lve.df.filter(f.col("studyStatistics.studyType") == "eqtl").select("variantId").distinct().count()
print(f"Number of unique variants in eqtl: {num_variants}")
num_variants = (
    lve.df.filter(f.col("studyStatistics.studyType") == "gwas-measurement").select("variantId").distinct().count()
)
print(f"Number of unique variants in gwas-measurement: {num_variants}")
num_variants = (
    lve.df.filter(f.col("studyStatistics.studyType") == "gwas-disease").select("variantId").distinct().count()
)
print(f"Number of unique variants in gwas-disease: {num_variants}")


Number of unique variants in cis-pqtl: 1260


Number of unique variants in eqtl: 75348


Number of unique variants in gwas-measurement: 18559
Number of unique variants in gwas-disease: 3854


## Sanity check


In [ ]:
lve.df.select(RescaledStatistics().estimated_beta, f.col("majorLdPopulationMaf.value").alias("MAF")).describe().show()


+-------+-------------------+-------------------+
|summary|      estimatedBeta|                MAF|
+-------+-------------------+-------------------+
|  count|             459516|             459516|
|   mean|0.06773143430478148| 0.2469594074979868|
| stddev| 0.9393964960986017|0.14857856426069216|
|    min|-2.9997936995577867|7.62779092394584E-6|
|    max| 2.9996285707276287|                0.5|
+-------+-------------------+-------------------+



In [ ]:
# Check the number of studyTypes after all filtering
from manuscript_methods import group_statistics

group_statistics(lve.df.select("studyStatistics.studyType"), [f.col("studyType")]).show()


+----------------+------+-----+-------------------+
|       studyType| count|    %|         percentage|
+----------------+------+-----+-------------------+
|            eqtl|354183|77.08|  77.07740318073799|
|gwas-measurement| 91147|19.84| 19.835435545225845|
|    gwas-disease| 12694| 2.76| 2.7624718181739047|
|        cis-pqtl|  1492| 0.32|0.32468945586225506|
+----------------+------+-----+-------------------+



In [ ]:
lve.df.show(1)


+--------------------+---------------+--------------------+--------------------+---------------+------------+---------------------+--------------------+-----------------+----------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------------------+------------------------+
|        studyLocusId|      variantId|             variant|             studyId|         geneId|originalBeta|originalStandardError|     locusStatistics|finemappingMethod|isTransQtl|       variantEffect|majorLdPopulation|majorLdPopulationMaf| majorLdPopulationAf|   variantStatistics|     studyStatistics|  rescaledStatistics|leadVariantConsequence|traitFromSourceMappedIds|
+--------------------+---------------+--------------------+--------------------+---------------+------------+---------------------+--------------------+-----------------+----------+--------------------+-----------------+--------------------+-----------

In [ ]:
lve.df.count()


459516

In [ ]:
lve.df.dropDuplicates(["variantId", "rescaledStatistics"]).count()


458825